# 08 — Content-Based Movie Recommendation

Goal: recommend movies that are similar to a selected movie.

Approach:
1. Combine descriptive text fields.
2. Convert the combined text into TF-IDF vectors.
3. Calculate cosine similarity.
4. Return the most similar movies.

This is a **content-based recommender**. It does not require user ratings or user history.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

movies = pd.read_csv("../data/processed/movies_cleaned.csv")
movies = movies.dropna(how="all").reset_index(drop=True)

text_columns = [
    "genre", "director", "writer",
    "star", "country", "company", "rating"
]

for col in text_columns + ["name"]:
    movies[col] = movies[col].fillna("Unknown").astype(str).str.strip()

# Repeat genre and key people to give them more influence in similarity.
movies["content"] = (
    movies["genre"] + " " + movies["genre"] + " " +
    movies["director"] + " " +
    movies["writer"] + " " +
    movies["star"] + " " +
    movies["country"] + " " +
    movies["company"] + " " +
    movies["rating"]
)

movies[["name", "content"]].head()

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(movies["content"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
# A title-to-index mapping.
# Duplicate titles are preserved; the first matching title is returned by this simple interface.
title_to_index = pd.Series(
    movies.index,
    index=movies["name"].str.lower()
).drop_duplicates()

def recommend_movies(title, n=10):
    title_key = str(title).strip().lower()

    if title_key not in title_to_index:
        matches = movies[
            movies["name"].str.contains(
                str(title),
                case=False,
                na=False,
                regex=False
            )
        ][["name", "year", "genre", "score"]].head(10)

        return None, matches

    idx = title_to_index[title_key]

    similarity_scores = linear_kernel(
        tfidf_matrix[idx:idx+1],
        tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[::-1]

    recommendations = movies.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    recommendations = recommendations[
        recommendations.index != idx
    ]

    columns = [
        "name", "year", "genre", "score",
        "director", "star", "similarity"
    ]

    return recommendations[columns].head(n), None

In [ ]:
recommendations, matches = recommend_movies("The Shining", n=10)

if recommendations is not None:
    recommendations
else:
    print("Title not found. Possible matches:")
    matches

In [ ]:
# Try another movie.
recommendations, matches = recommend_movies("Titanic", n=10)

if recommendations is not None:
    recommendations
else:
    print("Title not found. Possible matches:")
    matches

## Recommendation quality check

Look at several recommendations manually.

Ask:
- Do they share genre?
- Do they share directors, writers or stars?
- Do they have similar production context?
- Are some recommendations unexpectedly similar because of common generic text?

This is important because a recommender should be evaluated for usefulness, not only by whether the code runs.

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(tfidf, "../models/movie_tfidf_vectorizer.pkl")
joblib.dump(title_to_index, "../models/movie_title_index.pkl")

# Save the sparse TF-IDF matrix without converting it to a huge dense matrix.
from scipy import sparse
sparse.save_npz("../models/movie_tfidf_matrix.npz", tfidf_matrix)

movies.to_csv("../models/movie_recommendation_data.csv", index=False)

print("Saved recommendation artifacts in ../models/")

## Next application step

This notebook can later become a Flask recommendation endpoint.

Example user flow:

`User enters movie title → Flask finds movie → TF-IDF similarity → top recommendations → HTML page displays results`

The same idea can later be connected to the movie dashboard.